In [ ]:
# === Setup ===
# Runtime: <2m with OAI_FAST_MODE=1
# Hardware: CPU smoke; GPU recommended for full run
# Network: optional
# Competition-safe: No — learning profile
import os, random, math, re, json, csv, time
from pathlib import Path
import numpy as np
import torch

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
FAST_MODE = os.getenv("OAI_FAST_MODE", "0") == "1"
RUNTIME_PROFILE = os.getenv("OAI_RUNTIME_PROFILE", "cpu")
DEVICE = torch.device("cuda" if RUNTIME_PROFILE == "gpu" else "cpu")
if RUNTIME_PROFILE == "gpu" and not torch.cuda.is_available():
    raise RuntimeError("GPU profile requested but CUDA is unavailable")
torch.set_default_device(DEVICE)
_device_probe = (torch.ones(8, device=DEVICE) @ torch.ones(8, device=DEVICE)).item()
print(f"Compute device: {DEVICE}; probe={_device_probe:.1f}")
random.seed(42)
np.random.seed(42)
print(f"Runtime profile: {'fast' if FAST_MODE else 'full'}")

# Experiment — NMS threshold

**Hypothesis:** threshold cao giữ nhiều box trùng hơn.

In [ ]:
from torchvision.ops import nms

b = torch.tensor([[0.0, 0.0, 10.0, 10.0], [1.0, 1.0, 9.0, 9.0], [2.0, 2.0, 8.0, 8.0]])
s = torch.tensor([0.9, 0.8, 0.7])
print({t: nms(b, s, t).tolist() for t in (0.3, 0.5, 0.8)})

**Observation:** NMS threshold là trade-off duplicate detections và false suppression; phải tune trên validation theo mAP.